# Volume of Mixing vs Pressure

Computes ΔV_mix(P*) for 11 pressures P* = 1.0 – 2.0.

## Definition

$$\Delta V_{\rm mix} = V_{\rm mix} - V_{\rm pol,ref} - V_{\rm sol,ref}$$

**V_mix** — box volume of the isolated gel (`isolated_*.data`), i.e. the equilibrated slab after `isolate_gel.py` strips support, piston, and bath solvent. Contains $N_{\rm pol}$ polymer atoms and $N_{\rm sol}$ solvent atoms.

**Pure-phase references** — `polymer_pure` and `solvent_pure` NPT runs (100k steps, same P*) start from inputs trimmed by `split_gel.py` (8 % per face per species) to sample a homogeneous gel interior. Their equilibrated box volumes are $V_{\rm pol,eq}$ (from $N_{\rm pol,trim}$ atoms) and $V_{\rm sol,eq}$ (from $N_{\rm sol,trim}$ atoms).

Because $N_{\rm pol,trim} < N_{\rm pol}$ and $N_{\rm sol,trim} < N_{\rm sol}$ (trimming removes ~16 % of each species), the pure-phase volumes are scaled to the full isolated-gel atom count before subtracting:

$$V_{\rm pol,ref} = V_{\rm pol,eq} \times \frac{N_{\rm pol}}{N_{\rm pol,trim}}, \qquad V_{\rm sol,ref} = V_{\rm sol,eq} \times \frac{N_{\rm sol}}{N_{\rm sol,trim}}$$

The scaling factors and atom counts are written by `split_gel.py` to a JSON manifest (`*_volmix_manifest.json`) alongside the split data files and synced here by the cell below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

plt.rcParams.update({
    'font.family':        'CMU Serif',
    'mathtext.fontset':   'cm',
    'mathtext.rm':        'CMU Serif',
    'font.size':          20,
    'axes.titlesize':     22,
    'axes.labelsize':     25,
    'xtick.labelsize':    23,
    'ytick.labelsize':    23,
    'legend.fontsize':    23,
    'figure.titlesize':   22,
    'axes.unicode_minus': False,
    'figure.dpi':         120,
})

# --- Configuration ---
BASE_DATANAME  = "slab_support_5beads_tall_rho04"
INTERACTION    = "1.0_1.0"
PURE_INTER     = "1.0_0.0"
SLAB_STEPS     = 600000
PURE_STEPS     = 100000

DATA_DIR = Path("../../flow_data_local/volmix_sweep")

PRESSURES = [round(1.0 + i * 0.1, 1) for i in range(11)]
print(f"Pressures: {PRESSURES}")
print(f"Data root: {DATA_DIR.resolve()}")


In [ ]:
# === Sync volume data from Expanse ===
import paramiko, getpass, stat
from pathlib import Path

EXPANSE_HOST = "login.expanse.sdsc.edu"
EXPANSE_USER = "dpollard"
STAGE_DIR    = "/home/dpollard/Documents/lammps_runs/volmix_sweep/volmix_stage"

stage_script = (
    "SWEEP=~/Documents/lammps_runs/volmix_sweep\n"
    "DATA=~/Documents/lammps_data\n"
    "STAGE=${SWEEP}/volmix_stage\n"
    "mkdir -p \"$STAGE\"\n"
    "for P in 1.0 1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2.0; do\n"
    "  mkdir -p \"$STAGE/p${P}\"\n"
    "  SOL=$(ls -dt \"$SWEEP\"/solvent_*pstar${P}_* 2>/dev/null | head -1)\n"
    "  POL=$(ls -dt \"$SWEEP\"/polymer_*pstar${P}_* 2>/dev/null | head -1)\n"
    "  ISTEM=\"isolated_slab_support_5beads_tall_rho04_pstar${P}_1.0_1.0_600000\"\n"
    "  [ -n \"$SOL\" ] && cp \"$SOL\"/output_files/volume_data/box_dimensions_*.dat \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "  [ -n \"$POL\" ] && cp \"$POL\"/output_files/volume_data/box_dimensions_*.dat \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "  MANIFEST=\"$DATA/input_data/${ISTEM}_volmix_manifest.json\"\n"
    "  [ -f \"$MANIFEST\" ] && cp \"$MANIFEST\" \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "  cnt=$(ls \"$STAGE/p${P}/\" 2>/dev/null | wc -l)\n"
    "  echo \"  P=${P}: $cnt files staged\"\n"
    "done\n"
)

password = getpass.getpass(f"Expanse password for {EXPANSE_USER}: ")
totp     = getpass.getpass("TOTP / verification code: ")

def auth_handler(title, instructions, prompt_list):
    responses = []
    for prompt, echo in prompt_list:
        if "password" in prompt.strip().lower():
            responses.append(password)
        else:
            responses.append(totp)
    return responses

print("Connecting to Expanse...")
transport = paramiko.Transport((EXPANSE_HOST, 22))
transport.connect()
transport.auth_interactive(EXPANSE_USER, auth_handler)

ssh = paramiko.SSHClient()
ssh._transport = transport

print("Step 1 — staging files on Expanse...")
_, stdout, stderr = ssh.exec_command("bash -s", get_pty=False)
stdout.channel.sendall(stage_script.encode())
stdout.channel.shutdown_write()
print(stdout.read().decode())

print("Step 2 — downloading via SFTP (skips files already present)...")
sftp = ssh.open_sftp()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def sftp_download_dir(sftp, remote_dir, local_dir):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = f"{remote_dir}/{entry.filename}"
        local_path  = local_dir / entry.filename
        if stat.S_ISDIR(entry.st_mode):
            sftp_download_dir(sftp, remote_path, local_path)
        else:
            if local_path.exists() and local_path.stat().st_mtime >= entry.st_mtime:
                continue
            sftp.get(remote_path, str(local_path))

sftp_download_dir(sftp, STAGE_DIR, DATA_DIR)
sftp.close()
ssh.close()
print("Sync complete.")


## Volume parsing functions

In [ ]:
def avg_box_volume(path, skip_frac=0.5):
    """
    Parse box_dimensions_*.dat (columns: step lx ly lz).
    Returns time-averaged volume = mean(lx*ly*lz) over the last (1-skip_frac) fraction.
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 4:
                try:
                    step, lx, ly, lz = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
                    data.append(lx * ly * lz)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


def avg_pure_volume(path, skip_frac=0.0):
    """
    Parse vol_pure_*.dat (columns: step press_mean vol_mean rho_mean, block averages).
    Returns mean of vol_mean column over all blocks (skip_frac=0 since runs are short).
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    vol = float(parts[2])  # vol_mean column
                    data.append(vol)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


def load_manifest(path):
    """Load the volmix manifest JSON written by split_gel.py."""
    with open(path) as f:
        return json.load(f)


## Load volume data for each pressure

Uses the sweep manifest files written by pressure_sweep.sh to locate each run directory.

In [ ]:
import json

results = []
missing = []

for P in PRESSURES:
    pstr = f"{P:.1f}"
    dataname          = f"{BASE_DATANAME}_pstar{pstr}"
    isolated_dataname = f"isolated_{dataname}_{INTERACTION}_{SLAB_STEPS}"
    sol_dataname      = f"{isolated_dataname}_solvent_only"
    pol_dataname      = f"{isolated_dataname}_polymer_only"

    p_dir = DATA_DIR / f"p{pstr}"

    # Manifest written by split_gel.py — contains V_mix and scaling factors
    manifest_file = p_dir / f"{isolated_dataname}_volmix_manifest.json"

    # Pure-phase equilibrated box volumes
    sol_vol_file = p_dir / f"box_dimensions_{sol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"
    pol_vol_file = p_dir / f"box_dimensions_{pol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"

    if not all(f.exists() for f in [manifest_file, sol_vol_file, pol_vol_file]):
        missing.append(pstr)
        for label, fpath in [("manifest", manifest_file), ("solvent", sol_vol_file), ("polymer", pol_vol_file)]:
            if not fpath.exists():
                print(f"[SKIP] P*={pstr}: missing {label} → {fpath}")
        continue

    try:
        mf       = load_manifest(manifest_file)
        V_mix    = mf['V_mix_isolated']        # equilibrated isolated-gel box volume
        scale_pol = mf['scale_pol']             # N_pol_isolated / N_pol_trimmed
        scale_sol = mf['scale_sol']             # N_sol_isolated / N_sol_trimmed

        V_pol_eq = avg_box_volume(pol_vol_file, skip_frac=0.5)  # trimmed polymer NPT
        V_sol_eq = avg_box_volume(sol_vol_file, skip_frac=0.5)  # trimmed solvent NPT

        # Scale pure-phase volumes to the full isolated-gel atom count
        V_pol_ref = V_pol_eq * scale_pol
        V_sol_ref = V_sol_eq * scale_sol
        dV        = V_mix - V_pol_ref - V_sol_ref

        results.append({
            "P": P, "V_mix": V_mix, "V_sol": V_sol_ref,
            "V_pol": V_pol_ref, "dV_mix": dV,
            "scale_pol": scale_pol, "scale_sol": scale_sol,
        })
        print(f"P*={pstr}:  V_mix={V_mix:.2f}  V_sol_ref={V_sol_ref:.2f}  "
              f"V_pol_ref={V_pol_ref:.2f}  ΔV={dV:+.3f}  "
              f"(scale_pol={scale_pol:.3f} scale_sol={scale_sol:.3f})")
    except Exception as e:
        print(f"[ERROR] P*={pstr}: {e}")
        missing.append(pstr)

df = pd.DataFrame(results)
print(f"\nLoaded {len(df)}/{len(PRESSURES)} pressure points")
if missing:
    print(f"Missing: {missing}")


## Plot ΔV_mix vs P*

In [ ]:
if df.empty:
    print("No data to plot — run pressure_sweep.sh and wait for all jobs to complete.")
else:
    P      = df["P"].values
    V_mix  = df["V_mix"].values
    V_sol  = df["V_sol"].values
    V_pol  = df["V_pol"].values
    dV_mix = df["dV_mix"].values
    V_ref  = V_sol + V_pol        # isolated pure-component total

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: ΔV_mix / V_ref  (fractional volume of mixing) ---
    ax = axes[0]
    ax.plot(P, dV_mix / V_ref, "o-", color="steelblue", lw=2, ms=7)
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel(r"$P^*$")
    ax.set_ylabel(r"$\Delta V_{\rm mix}\;/\;(V_{\rm sol}+V_{\rm pol})$")
    ax.set_title("Volume of Mixing")
    ax.grid(True, alpha=0.3)

    # --- Right: component volumes / V_mix  (volume fractions) ---
    ax2 = axes[1]
    ax2.plot(P, V_sol / V_mix * 100, "s--", label=r"$V_{\rm solvent}/V_{\rm mix}$", color="tomato",   lw=1.5, ms=5)
    ax2.plot(P, V_pol / V_mix * 100, "^--", label=r"$V_{\rm polymer}/V_{\rm mix}$", color="seagreen",  lw=1.5, ms=5)
    ax2.set_xlabel(r"$P^*$")
    ax2.set_ylabel(r"$V_i\;/\;V_{\rm mix}\;[\%]$")
    ax2.set_title("Volume Fractions (isolated gel)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    PLOT_DIR = Path("../../flow_data_local/plots/volmix")
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(PLOT_DIR / "volume_of_mixing.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOT_DIR / 'volume_of_mixing.png'}")


## Summary table

In [ ]:
if not df.empty:
    from IPython.display import display
    display_df = df.copy()
    display_df.columns = ["P*", "V_mix [σ³]", "V_solvent [σ³]", "V_polymer [σ³]", "ΔV_mix [σ³]"]
    display_df = display_df.round(3)
    display(display_df)
